In [2]:
import json
import os

import tiktoken
from glob import glob
import pandas as pd

In [3]:
def load_model_answers(answer_dir: str):
    """Load model answers.

    The return value is a python dict of type:
    Dict[model_name: str -> Dict[question_id: int -> answer: dict]]
    """
    filenames = glob(os.path.join(answer_dir, "*.jsonl"))
    #print(filenames)
    filenames.sort()
    model_answers = {}

    for filename in filenames:
        model_name = os.path.basename(filename)[:-6]
        answer = {}
        with open(filename) as fin:
            for line in fin:
                line = json.loads(line)
                answer[line["question_id"]] = line
        model_answers[model_name] = answer

    return model_answers

In [4]:
def load_questions_dict(question_file: str):
    """Load questions from a file."""
    questions = []
    with open(question_file, "r") as ques_file:
        for line in ques_file:
            if line:
                questions.append(json.loads(line))
    return questions

In [17]:
def tokenize_and_sum(questions, model_answers, tokenizer):
    """Tokenize and sum tokens for questions and answers."""
    token_data = []
    for model_name, answers in model_answers.items():
        for question in questions:
            q_id = question['question_id']
            if q_id in answers:
                # Tokenize the question
                question_tokens = tokenizer.encode(question['turns'][0]['content'])
                question_token_len = len(question_tokens)
                
                # Get the token length from the answer's token_len field
                answer_token_len = answers[q_id]['choices'][0]['turns'][0]['token_len']
                
                # Sum the question and answer tokens
                total_tokens = question_token_len + answer_token_len
                token_data.append((model_name, q_id, total_tokens))
    return token_data

In [18]:
def create_token_stats_dataframe(token_data):
    """Create a DataFrame with statistics on the token data."""
    df = pd.DataFrame(token_data, columns=['model', 'question_id', 'total_tokens'])

    # Aggregate statistics
    stats = df.groupby('model')['total_tokens'].agg(
        max_tokens='max',
        avg_tokens='mean',
        over_1000=lambda x: (x > 1000).sum(),
        over_1500=lambda x: (x > 1500).sum(),
        over_2000=lambda x: (x > 2000).sum(),
        over_2500=lambda x: (x > 2500).sum(),
        over_3000=lambda x: (x > 3000).sum(),
        over_3500=lambda x: (x > 3500).sum(),
        over_4000=lambda x: (x > 4000).sum(),
        over_4500=lambda x: (x > 4500).sum(),
        over_5000=lambda x: (x > 5000).sum()
    ).reset_index()

    return stats


In [19]:
# Load the questions and model answers
question_file = 'arena-hard-auto/data/arena-hard-v0.1/question.jsonl'
answer_dir = 'arena-hard-auto/data/arena-hard-v0.1/model_answer'

questions = load_questions_dict(question_file)
model_answers = load_model_answers(answer_dir)

print(model_answers.keys())

dict_keys(['claude-3-5-sonnet-20240620', 'claude-3-opus-20240229', 'dbrx-instruct-preview', 'gemini-1.5-flash-api-0514', 'gemini-1.5-pro-api-0514', 'gemma-2-27b-it', 'gpt-3.5-turbo-0125', 'gpt-4-0314', 'gpt-4-0613-compare', 'gpt-4-0613', 'gpt-4-1106-preview', 'gpt-4-turbo-2024-04-09', 'gpt-4o-2024-08-06', 'gpt-4o-mini-compare', 'gpt-4o-mini', 'gpt-4o', 'gpt-4o_no_prompt', 'gpt-4o_old', 'llama-3.1-405b-instruct', 'llama-3.1-70b-instruct-compare', 'llama3_1_70b', 'llama3_1_70b_awq4', 'llama3_1_70b_awq_int4', 'llama3_1_70b_awq_int4_no_prompt', 'llama3_1_70b_fp8', 'llama3_1_70b_fp8_1', 'llama3_1_70b_fp8_2', 'llama3_1_70b_fp8_no_prompt', 'llama3_1_70b_no_prompt', 'llama3_1_70b_no_prompt_2', 'llama3_1_8b', 'llama3_1_8b_fp8', 'llama3_70b_awq_int4_no_prompt', 'llama3_70b_fp8_no_prompt', 'llama3_70b_no_prompt', 'llama3_8b', 'llama3_8b_fp8', 'mistral-7b-instruct', 'mistral-large-2407', 'mixtral-8x22b-instruct-v0.1', 'mixtral-8x7b-instruct-v0.1', 'snowflake-arctic-instruct'])


In [20]:
print(questions[0])

{'question_id': '328c149ed45a41c0b9d6f14659e63599', 'category': 'arena-hard-v0.1', 'cluster': 'ABC Sequence Puzzles & Groups', 'turns': [{'content': 'Use ABC notation to write a melody in the style of a folk tune.'}]}


In [21]:
print(model_answers['gpt-4o']['0901d02592b347d8aa2cb99b02bf0dad'])

{'question_id': '0901d02592b347d8aa2cb99b02bf0dad', 'answer_id': 'iKMvh7icV9FbRxVZeo9uiJ', 'model_id': 'gpt-4o', 'choices': [{'index': 0, 'turns': [{'content': 'Understood. Please provide the message you would like me to evaluate.', 'token_len': 14}]}], 'tstamp': 1723127480.6005936}


In [22]:
# Tokenizer for GPT-4 (adjust as needed for your model)
tokenizer = tiktoken.encoding_for_model("gpt-4o")

# Tokenize and sum tokens for each question + answer pair
token_data = tokenize_and_sum(questions, model_answers, tokenizer)

In [23]:
# Create the DataFrame with aggregated statistics
token_stats_df = create_token_stats_dataframe(token_data)

token_stats_df

,model,max_tokens,avg_tokens,over_1000,over_1500,over_2000,over_2500,over_3000,over_3500,over_4000,over_4500,over_5000
0,claude-3-5-sonnet-20240620,3554,662.084000,49,12,7,2,2,1,0,0,0
1,claude-3-opus-20240229,2977,636.672000,41,13,5,2,0,0,0,0,0
2,dbrx-instruct-preview,4026,509.946000,31,10,6,2,1,1,1,0,0
3,gemini-1.5-flash-api-0514,2696,737.852000,85,19,5,2,0,0,0,0,0
4,gemini-1.5-pro-api-0514,3737,771.408000,98,18,6,2,1,1,0,0,0
5,gemma-2-27b-it,4748,672.600000,48,12,4,1,1,1,1,1,0
6,gpt-3.5-turbo-0125,3996,424.650000,16,7,6,3,2,2,0,0,0
7,gpt-4-0314,2694,518.184000,29,6,4,1,0,0,0,0,0
8,gpt-4-0613,2741,452.445183,12,3,2,1,0,0,0,0,0
9,gpt-4-0613-compare,2640,449.808000,16,6,4,1,0,0,0,0,0


# Show Judgement Tokens

In [6]:
# Tokenizer for GPT-4 (adjust as needed for your model)
tokenizer = tiktoken.encoding_for_model("gpt-4o")

In [7]:
judgement_json = load_questions_dict('arena-hard-auto/data/arena-hard-v0.1/model_judgment/gpt-4o/gpt-4o.jsonl')

In [8]:
print(judgement_json[0])

{'question_id': '328c149ed45a41c0b9d6f14659e63599', 'model': 'gpt-4o', 'judge': 'gpt-4o', 'games': [{'user_prompt': '<|User Prompt|>\nUse ABC notation to write a melody in the style of a folk tune.\n\n<|The Start of Assistant A\'s Answer|>\nX:1\nT:Untitled Folk Tune\nM:4/4\nL:1/8\nK:G\n|:G2A2|B2A2|G2E2|D4|E2F2|G2F2|E2C2|B,4|\nA2B2|c2B2|A2F2|E4|D2E2|F2E2|D2B,2|C4:|\n<|The End of Assistant A\'s Answer|>\n\n<|The Start of Assistant B\'s Answer|>\nSure! ABC notation is a way of writing music using the letters A through G, along with other symbols to indicate rhythm, key, and other musical elements. Here is a simple melody in the style of a folk tune using ABC notation:\n\n```abc\nX:1\nT:Simple Folk Tune\nM:4/4\nL:1/8\nK:G\nD2 | "G"G2 G2 A2 B2 | "C"c2 B2 A2 G2 | "D"F2 F2 E2 D2 | "G"G4 D2 |\n"G"G2 G2 A2 B2 | "C"c2 B2 A2 G2 | "D"F2 F2 E2 D2 | "G"G4 z2 ||\n"G"B2 d2 d2 B2 | "C"c2 B2 A2 G2 | "D"F2 F2 E2 D2 | "G"G4 D2 |\n"G"G2 G2 A2 B2 | "C"c2 B2 A2 G2 | "D"F2 F2 E2 D2 | "G"G4 z2 |\n```\n\nExplan

In [9]:
print(judgement_json[0]['games'][0]['judgment'])


Here is my response to the user prompt:

```abc
X:1
T:Folk Melody
M:4/4
L:1/8
K:G
|: G2 A2 B2 G2 | E2 D2 G2 E2 | D2 E2 F2 D2 | G4 D4 :|
|: B2 d2 d2 B2 | c2 B2 A2 G2 | F2 E2 D2 E2 | G4 z4 :|
```

Explanation:
- `X:1` is the reference number of the tune.
- `T:Folk Melody` is the title of the tune.
- `M:4/4` indicates the time signature (4/4 time).
- `L:1/8` indicates the default note length (eighth notes).
- `K:G` indicates the key of G major.
- The letters A-G represent the notes, with numbers indicating the duration (e.g., `G2` is a G note held for two beats).
- The vertical bars `|` separate measures.
- The double bar `||` indicates the end of a section.
- The colon `:` indicates repetition of the section.

This melody is designed to be simple and repetitive, capturing the essence of a traditional folk tune.

Now, let's evaluate the responses from Assistant A and Assistant B.

### Evaluation of Assistant A's Answer:
1. **Correctness**: The ABC notation provided by Assistant A is corr

In [10]:
print(judgement_json[0]['games'][0]['user_prompt'])

<|User Prompt|>
Use ABC notation to write a melody in the style of a folk tune.

<|The Start of Assistant A's Answer|>
X:1
T:Untitled Folk Tune
M:4/4
L:1/8
K:G
|:G2A2|B2A2|G2E2|D4|E2F2|G2F2|E2C2|B,4|
A2B2|c2B2|A2F2|E4|D2E2|F2E2|D2B,2|C4:|
<|The End of Assistant A's Answer|>

<|The Start of Assistant B's Answer|>
Sure! ABC notation is a way of writing music using the letters A through G, along with other symbols to indicate rhythm, key, and other musical elements. Here is a simple melody in the style of a folk tune using ABC notation:

```abc
X:1
T:Simple Folk Tune
M:4/4
L:1/8
K:G
D2 | "G"G2 G2 A2 B2 | "C"c2 B2 A2 G2 | "D"F2 F2 E2 D2 | "G"G4 D2 |
"G"G2 G2 A2 B2 | "C"c2 B2 A2 G2 | "D"F2 F2 E2 D2 | "G"G4 z2 ||
"G"B2 d2 d2 B2 | "C"c2 B2 A2 G2 | "D"F2 F2 E2 D2 | "G"G4 D2 |
"G"G2 G2 A2 B2 | "C"c2 B2 A2 G2 | "D"F2 F2 E2 D2 | "G"G4 z2 |
```

Explanation:
- `X:1` is the reference number of the tune.
- `T:Simple Folk Tune` is the title of the tune.
- `M:4/4` indicates the time signature (4/4 tim

In [19]:
def judgement_toks(judgements, tokenizer):
    """Tokenize and sum tokens for questions and answers."""
    max_input_toks = 0
    max_output_toks = 0
    max_total_toks = 0


    for judgement in judgements:

        for game in judgement['games']:
            user_prompt = game['user_prompt']
            judgment = game['judgment']

            # Tokenize the user prompt
            user_prompt_tokens = tokenizer.encode(user_prompt)
            user_prompt_token_len = int(len(user_prompt_tokens))

            # Tokenize the judgment
            judgment_tokens = tokenizer.encode(judgment)
            judgment_token_len = int(len(judgment_tokens))

            # Sum the question and answer tokens
            total_tokens = int(user_prompt_token_len + judgment_token_len)

            if user_prompt_token_len > max_input_toks:
                max_input_toks = user_prompt_token_len

            if judgment_token_len > int(max_output_toks):
                max_output_toks = judgment_token_len

            if total_tokens > max_total_toks:
                max_total_toks = total_tokens

    
    print(20*"=" + "Tokens for Judgements" + 20*"=" + "\n") 
    print(f"Max input tokens: {max_input_toks}")
    print(f"Max output tokens: {max_output_toks}")
    print(f"Max total tokens: {max_total_toks}")


    token_data = [max_input_toks, max_output_toks, max_total_toks]

    return token_data

In [20]:
judgement_toks(judgement_json, tokenizer)


====================Tokens for Judgements====================

Max input tokens: 4471
Max output tokens: 8193
Max total tokens: 12524


[4471, 8193, 12524]